# Embeddings - Hands-On Tour

A single notebook walking through `../docs/` end to end: generate an
embedding &rarr; measure similarity &rarr; chunk a document &rarr; build a
search engine &rarr; cluster &rarr; deduplicate &rarr; classify &rarr; evaluate.
Everything runs against a **local Ollama embedding model** - no API
key, no cost, nothing leaves your machine, so anyone can run this
exactly as written.

## Setup

```bash
ollama pull nomic-embed-text
pip install requests numpy scikit-learn
```

In [ ]:
import requests
import numpy as np

OLLAMA_URL = "http://localhost:11434/api/embeddings"
MODEL = "nomic-embed-text"

def embed(text: str) -> np.ndarray:
    r = requests.post(OLLAMA_URL, json={"model": MODEL, "prompt": text})
    return np.array(r.json()["embedding"])

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

## 1. Generate your first embedding

See `docs/01-What-Are-Embeddings.md`. The same input always produces
the same vector - no sampling, unlike a chat model.

In [ ]:
vector = embed("The pod is stuck in CrashLoopBackOff")
print(f"dimensions: {len(vector)}")
print(f"first 5 numbers: {vector[:5]}")

## 2. Similarity: related vs. unrelated text

See `docs/05-Similarity-Metrics.md`. Cosine similarity measures
direction, not magnitude - the standard default for text.

In [ ]:
a = embed("The pod is stuck in CrashLoopBackOff")
b = embed("Container keeps restarting and failing health checks")
c = embed("Best pizza toppings for a party")

print(f"related pair:   {cosine_similarity(a, b):.4f}")
print(f"unrelated pair: {cosine_similarity(a, c):.4f}")

## 3. Chunking before embedding

See `docs/07-Chunking-Before-Embedding.md`. A whole document's
embedding is a blurry average - chunking preserves granularity.

In [ ]:
def chunk_fixed(text, chunk_size=120, overlap=25):
    chunks, start = [], 0
    while start < len(text):
        chunks.append(text[start:start + chunk_size])
        start += chunk_size - overlap
    return chunks

runbook = ("To debug CrashLoopBackOff: check kubectl describe pod, "
           "then kubectl logs --previous for the crash reason. Common "
           "causes are OOMKilled, a failed liveness probe, or a bad image.")

chunks = chunk_fixed(runbook)
for i, c in enumerate(chunks):
    print(f"chunk {i}: {c!r}")

## 4. Build a semantic search engine from scratch

See `docs/10-Building-Semantic-Search-From-Scratch.md`. Embed
documents, embed the query, rank by cosine similarity - that's the
whole algorithm.

In [ ]:
class SimpleSearchIndex:
    def __init__(self):
        self.entries = []

    def add(self, text, metadata=None):
        self.entries.append({"text": text, "vector": embed(text), "metadata": metadata or {}})

    def search(self, query, top_k=3):
        q = embed(query)
        scored = [{**e, "score": cosine_similarity(q, e["vector"])} for e in self.entries]
        scored.sort(key=lambda x: x["score"], reverse=True)
        return scored[:top_k]

index = SimpleSearchIndex()
index.add("To debug CrashLoopBackOff, check kubectl describe pod and logs --previous.", {"source": "runbook-01.md"})
index.add("When disk usage hits 90% on a primary DB, it's HIGH severity per SRE-042.", {"source": "runbook-02.md"})
index.add("To roll back a failed deployment, run kubectl rollout undo.", {"source": "runbook-03.md"})
index.add("Best pizza toppings for a team lunch order.", {"source": "random-doc.md"})

for r in index.search("one of our pods keeps restarting, what do I check?", top_k=2):
    print(f"[{r['score']:.3f}] {r['metadata']['source']}: {r['text']}")

## 5. Clustering: find groups with no labels

See `docs/12-Clustering-With-Embeddings.md`.

In [ ]:
from sklearn.cluster import KMeans

titles = [
    "checkout service returning 500s",
    "payment API throwing 500 errors",
    "disk full on db-primary-02",
    "database disk usage critical",
    "login page slow to load",
    "auth service high latency",
]
vectors = np.array([embed(t) for t in titles])
kmeans = KMeans(n_clusters=3, n_init="auto", random_state=42).fit(vectors)

for title, cluster_id in zip(titles, kmeans.labels_):
    print(f"cluster {cluster_id}: {title}")

## 6. Deduplication: same event, different words

See `docs/13-Deduplication-With-Embeddings.md`. Flag candidates for
review - never auto-merge on similarity score alone.

In [ ]:
tickets = [
    "Disk usage critical on db-primary-02",
    "db-primary-02 is almost out of disk space",
    "Checkout service returning 500 errors",
]
vectors = [embed(t) for t in tickets]

for i in range(len(tickets)):
    for j in range(i + 1, len(tickets)):
        score = cosine_similarity(vectors[i], vectors[j])
        if score >= 0.80:
            print(f"[{score:.3f}] possible duplicate:\n  {tickets[i]}\n  {tickets[j]}\n")

## 7. Classification without an LLM call

See `docs/14-Classification-With-Embeddings.md`. Nearest-neighbor
comparison against a handful of labeled examples.

In [ ]:
labeled = [
    ("CPU at 45% on web-node-01", "LOW"),
    ("Disk at 98% on db-primary-01, 5 minutes to full", "HIGH"),
    ("Memory at 78% on cache-node-03", "MEDIUM"),
]
labeled_vectors = [(embed(t), label) for t, label in labeled]

def classify(text):
    q = embed(text)
    best_label, best_score = None, -1
    for v, label in labeled_vectors:
        s = cosine_similarity(q, v)
        if s > best_score:
            best_label, best_score = label, s
    return best_label, best_score

print(classify("Disk at 95%, 10 minutes to full on the primary"))

## 8. Evaluate search quality with recall@k

See `docs/16-Evaluating-Embedding-Quality.md`. "It looks reasonable"
isn't good enough - measure it.

In [ ]:
eval_set = [
    {"query": "how do I restart a stuck pod?", "expected_source": "runbook-01.md"},
    {"query": "what happens when disk is full on the primary db?", "expected_source": "runbook-02.md"},
]

def recall_at_k(index, eval_set, k=3):
    hits = 0
    for case in eval_set:
        sources = [r["metadata"]["source"] for r in index.search(case["query"], top_k=k)]
        hits += case["expected_source"] in sources
    return hits / len(eval_set)

print(f"recall@2: {recall_at_k(index, eval_set, k=2):.0%}")

## Wrap-up

| Section | Concept | Docs chapter |
| --- | --- | --- |
| 1 | Generating embeddings | 01 |
| 2 | Similarity metrics | 05 |
| 3 | Chunking | 07 |
| 4 | Semantic search | 10 |
| 5 | Clustering | 12 |
| 6 | Deduplication | 13 |
| 7 | Classification | 14 |
| 8 | Evaluation | 16 |

Not covered here but worth running from `../examples/`:
`04_clean_before_embedding.py` (stripping log noise) and
`09_hybrid_search.py` (blending semantic + keyword search).

Next: `04-Vector-Databases` - the same search engine from section 4,
scaled past a Python list.